In [1]:
from utils.init_db import wipe_db
from pipeline_orchestration import full_pipeline_run
import duckdb
from pathlib import Path
import pandas as pd
import os
flag = 0

In [2]:

# Change the current working directory to the project root
if flag == 0:
    NOTEBOOK_DIR = Path.cwd()
    PROJECT_ROOT = NOTEBOOK_DIR.parent
    os.chdir(PROJECT_ROOT)
    print(f"Notebook working directory set to project root: {Path.cwd()}")
    flag += 1
else:
    print(f"Notebook working directory set to project root: {Path.cwd()}")

Notebook working directory set to project root: c:\dev\colibri-coding-task


In [3]:
# wipe all data to make sure we are starting with a clean slate.
wipe_db(wipe_data=True, wipe_layer="all")

RUNNING: wipe_db 23:29:30.577
Deleted directory: data/b_bronze/data_group_1
Deleted directory: data/b_bronze/data_group_2
Deleted directory: data/b_bronze/data_group_3
Deleted file: data/b_bronze/ingestion_metadata.csv
Deleted directory: data/c_silver/silver_data_quality_report
Deleted directory: data/c_silver/turbine_clean
Deleted directory: data/d_gold/turbine_summary
EXECUTED: wipe_db 23:29:30.662



In [4]:
# Initial Pipeline run - single entry point to run the entire pipeline.
full_pipeline_run()

RUNNING: initialise_db 23:29:30.679
INFO: Created metadata schema if it did not exist
INFO: Created analytics schema if it did not exist
INFO: Created bronze processing state table if it did not exist
INFO: Created silver processing state table for turbine summary if it did not exist
INFO: Created silver processing state table for anomaly detection if it did not exist
EXECUTED: initialise_db 23:29:30.715

RUNNING: land_turbine_data 23:29:30.715
INFO: Ingesting turbine data for data group 1 for 10/03/2026 ...
INFO: Ingesting turbine data for data group 2 for 10/03/2026 ...
INFO: Ingesting turbine data for data group 3 for 10/03/2026 ...
EXECUTED: land_turbine_data 23:29:30.715

RUNNING: ingest_turbine_data 23:29:30.715
Data written to data/b_bronze/data_group_1/data_group_1_20260310_232930.parquet
Data written to data/b_bronze/data_group_2/data_group_2_20260310_232930.parquet
Data written to data/b_bronze/data_group_3/data_group_3_20260310_232930.parquet
Data written to data/b_bronze/in

In [5]:
# Now add more data to each of the csvs

# The new data has the data for each turbine for 2022/04/01, but with the following values affected by missing data or outliers:

    # timestamp                  turbine_id        reason
    # 2022-04-01 00:00:00        1                 missing
    # 2022-04-01 18:00:00        2                 missing
    # 2022-04-01 15:00:00        3                 outlier
    # 2022-04-01 06:00:00        6                 outlier
    # 2022-04-01 10:00:00        9                 missing
    # 2022-04-01 15:00:00        10                missing
    # 2022-04-01 06:00:00        15                missing
    # 2022-04-01 15:00:00        13                missing
    # 2022-04-01 15:00:00        11                outlier


for data_group in [1, 2, 3]:
    df_base = pd.read_csv(f"data/mock_extra/original/data_group_{data_group}.csv")
    df_extra = pd.read_csv(f"data/mock_extra/data_group_{data_group}_extra.csv")

    df_union = pd.concat([df_base, df_extra], ignore_index=True)

    # print(len(df_base), len(df_extra), len(df_union))
    # Save to a location
    df_union.to_csv(f"data/a_raw/data_group_{data_group}.csv", index=False)
print("NEW BATCH OF BAD DATA ADDED")


NEW BATCH OF BAD DATA ADDED


In [6]:
# Check the turbine summary data

temp_con = duckdb.connect("db/windfarm.duckdb")
cleaned_data_df = temp_con.execute("SELECT * FROM 'data/c_silver/turbine_clean/*.parquet';").df()
summary_data_df = temp_con.execute("SELECT * FROM 'data/d_gold/turbine_summary/*.parquet';").df()

print(cleaned_data_df)
print(summary_data_df)
temp_con.close()

       turbine_id           timestamp  wind_speed  wind_direction  \
0               1 2022-03-01 00:00:00        11.8             169   
1               1 2022-03-01 01:00:00        11.6             152   
2               1 2022-03-01 02:00:00        13.8              73   
3               1 2022-03-01 03:00:00        10.5              61   
4               1 2022-03-01 04:00:00         9.1             209   
...           ...                 ...         ...             ...   
11155          10 2022-03-31 19:00:00        11.4              23   
11156          10 2022-03-31 20:00:00        12.3              26   
11157          10 2022-03-31 21:00:00        10.0             115   
11158          10 2022-03-31 22:00:00        14.0              82   
11159          10 2022-03-31 23:00:00        10.1              19   

       power_output       source_file  \
0               2.7  data_group_1.csv   
1               4.4  data_group_1.csv   
2               2.9  data_group_1.csv   
3      

In [ ]:
# Re-run the pipeline to see the incremental loading work.
import time
time.sleep(2)  # Sleep for a bit to ensure file system has registered the new files
full_pipeline_run()

RUNNING: initialise_db 23:29:31.418
INFO: Created metadata schema if it did not exist
INFO: Created analytics schema if it did not exist
INFO: Created bronze processing state table if it did not exist
INFO: Created silver processing state table for turbine summary if it did not exist
INFO: Created silver processing state table for anomaly detection if it did not exist
EXECUTED: initialise_db 23:29:31.450

RUNNING: land_turbine_data 23:29:31.450
INFO: Ingesting turbine data for data group 1 for 10/03/2026 ...
INFO: Ingesting turbine data for data group 2 for 10/03/2026 ...
INFO: Ingesting turbine data for data group 3 for 10/03/2026 ...
EXECUTED: land_turbine_data 23:29:31.450

RUNNING: ingest_turbine_data 23:29:31.450
Data written to data/b_bronze/data_group_1/data_group_1_20260310_232931.parquet
Data written to data/b_bronze/data_group_2/data_group_2_20260310_232931.parquet
Data written to data/b_bronze/data_group_3/data_group_3_20260310_232931.parquet
Data written to data/b_bronze/in

c:\dev\colibri-coding-task\src\c_silver\transforms.py:30: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  missing["issue_type"] = "missing_record"
c:\dev\colibri-coding-task\src\c_silver\transforms.py:64: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  outliers["issue_type"] = "outlier"
c:\dev\colibri-coding-task\src\c_silver\transforms.py:30: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in th

In [8]:
# Check the turbine summary data

temp_con = duckdb.connect("db/windfarm.duckdb")
cleaned_data_df = temp_con.execute("SELECT * FROM 'data/c_silver/turbine_clean/*.parquet' where timestamp > '2022-03-31 23:59:59';").df()
summary_data_df = temp_con.execute("SELECT * FROM 'data/d_gold/turbine_summary/*.parquet' where date > '2022-03-31';").df()
dq_df = temp_con.execute("SELECT * FROM analytics.silver_data_quality_report;").df()

# Notice how we now have data from the new upload and data quality flags from the new data
print(cleaned_data_df)
print(summary_data_df)
print(dq_df)
temp_con.close()

     turbine_id           timestamp  wind_speed  wind_direction  power_output  \
0             2 2022-04-01 00:00:00        11.6            24.0           2.2   
1             2 2022-04-01 01:00:00        12.8            35.0           4.2   
2             2 2022-04-01 02:00:00         9.9           103.0           3.8   
3             2 2022-04-01 03:00:00        12.4           274.0           3.4   
4             2 2022-04-01 04:00:00        14.9           335.0           2.0   
..          ...                 ...         ...             ...           ...   
355          15 2022-04-01 19:00:00        12.5            61.0           3.2   
356          15 2022-04-01 20:00:00        11.8           215.0           4.0   
357          15 2022-04-01 21:00:00        11.0           201.0           4.2   
358          15 2022-04-01 22:00:00         9.0           193.0           2.7   
359          15 2022-04-01 23:00:00        11.0           350.0           2.6   

          source_file      

In [9]:
dq_df

,turbine_id,timestamp,wind_speed,wind_direction,power_output,source_file,file,issue_type,mean,std,zscore
0,2,2022-04-01 18:00:00,NaN,NaN,NaN,None,None,missing_record,NaN,NaN,NaN
1,1,2022-04-01 00:00:00,NaN,NaN,NaN,None,None,missing_record,NaN,NaN,NaN
2,9,2022-04-01 10:00:00,NaN,NaN,NaN,None,None,missing_record,NaN,NaN,NaN
3,10,2022-04-01 15:00:00,NaN,NaN,NaN,None,None,missing_record,NaN,NaN,NaN
4,13,2022-04-01 15:00:00,NaN,NaN,NaN,None,None,missing_record,NaN,NaN,NaN
5,15,2022-04-01 06:00:00,NaN,NaN,NaN,None,None,missing_record,NaN,NaN,NaN
6,3,2022-04-01 15:00:00,9.2,171.0,500.0,data_group_1.csv,data\b_bronze\data_group_1\data_group_1_202603...,outlier,23.508333,101.496094,4.694680
7,6,2022-04-01 06:00:00,9.2,97.0,500.0,data_group_2.csv,data\b_bronze\data_group_2\data_group_2_202603...,outlier,23.633333,101.469521,4.694677
8,11,2022-04-01 15:00:00,9.7,135.0,500.0,data_group_3.csv,data\b_bronze\data_group_3\data_group_3_202603...,outlier,23.679167,101.460236,4.694655
